In [1]:
import sys

In [1]:
from radprocess.pipeline.Pipeline import Pipeline

In [ ]:
pipe = Pipeline() # create instance

In [ ]:
# Step 0: point to the RAMSES simulation output and to the postprocessing output folder.
ramses_path = '/data/pebbles/scratch/outputs/Mass500/output_00940'
pipe.configparams.dir.ramses_output = ramses_path

pipe_path = '/data/pebbles/scratch/sacha/enygma/examples/mass500/subboxes/test1/'
pipe.configparams.dir.pipeline_output = pipe_path


In [ ]:
# Check all configuration parameters
pipe.configparams

In [ ]:
# read hydro_descriptor file
print(pipe.read_hydro_descriptor())

In [1]:
# Inspect sinks
pipe.read_sink_info()

In [ ]:
# Step 0b: update pymsesrc
pipe.set_pymsesrc()

In [ ]:
# Step 1: Load RAMSES
pipe.load_ramses()

In [ ]:
# Step 2: POLARIS grid
pipe.convert_to_polaris()

In [ ]:
# Step 3: RADMC-3D grid
pipe.convert_to_radmc()

In [ ]:
# Step 4: POLARIS opacity run
dust = [
    {"path": "/path/silicate.cs", "weight": 0.625},
    {"path": "/path/carbon.cs",   "weight": 0.375},
]

#pipe.configparams.polaris.dust_size_min = 1e-8

pipe.run_polaris_opacity(dust_components=dust, 
                         dust_size_min=5e-9,
                         dust_size_max=2.5e-7
                        )


In [ ]:
# Step 5: Prepare RADMC-3D inputs
pipe.prepare_radmc3d_inputs(nphot=1_000_000, setthreads=8)

In [ ]:
# set radmc3d control parameters (radmc3d.inp):
pipe.configparams.radmc3d.nphot = 10000000
pipe.configparams.radmc3d.setthreads = 16


In [ ]:
# Step 6
temp_file = pipe.run_radmc3d_mctherm()
# temp_file is now Path("{working_dir}/radmc3d/dust_temperature.bdat")

In [ ]:
# Step 7
merged_grid = pipe.merge_temperature()
# merged_grid is Path("{working_dir}/polaris/grid_temp.radmc3d.dat")

In [ ]:
# Step 8
# Full-box imaging
pipe.render_images(
    dust_components=[
        {"path": "/path/silicate.cs", "weight": 0.625},
        {"path": "/path/carbon.cs",   "weight": 0.375},
    ],
    npix=512,
    distance_pc=140.0,
    wavelengths_mm=[0.87, 1.3, 3.0],
    label="whole",
)

# Zoomed inner imaging (same call, different params)
pipe.render_images(
    dust_components=[...],
    npix=512,
    distance_pc=140.0,
    wavelengths_mm=[0.87, 1.3, 3.0],
    midplane_zoom=10,
    fov_m=some_value,   # grid_size / 10
    label="inner",
)